Ingest the CSV dataset
SOURCE: tax_obligations_source.csv
TARGET: tx_dev.bronze.brz_tax_obligations

1.Configure the source and target

In [0]:
from datetime import datetime, timezone
import uuid

from pyspark.sql import functions as F

SOURCE_PATH = (
    "/Volumes/tx_dev/bronze/source_files/"
    "tax_obligations_source.csv"
)

TARGET_TABLE = "tx_dev.bronze.brz_tax_obligations"
AUDIT_TABLE = "tx_dev.control.tax_ingestion_audit"

SOURCE_SYSTEM = "TAX_REPORTING"
LOAD_TYPE = "FULL"
DRY_RUN = False

print("Tax Obligations Bronze Ingestion")

2.Read the CSV

In [0]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SOURCE_PATH)
)

print(f"Source record count: {df.count()}")

display(df)

3.Standardize column names

In [0]:
import re

def standardize_column_name(column_name):
    column_name = column_name.strip()
    column_name = re.sub(r"[^a-zA-Z0-9]+", "_", column_name)
    return column_name.strip("_").lower()

for old_name in df.columns:
    df = df.withColumnRenamed(
        old_name,
        standardize_column_name(old_name)
    )

display(df)

4.Add ingestion metadata(METADATA COLUMN)

In [0]:
df = (
    df
    .withColumn("source_file_name", F.lit(SOURCE_PATH.split("/")[-1]))
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("ingestion_date", F.current_date())
)

display(df)

Validate required fields

In [0]:
required_columns = [
    "obligation_id",
    "tax_entity_id",
    "tax_type",
    "tax_year",
    "reporting_period"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

null_condition = F.lit(False)

for column in required_columns:
    null_condition = null_condition | F.col(column).isNull()

invalid_df = df.filter(null_condition)

print(f"Total records: {df.count()}")
print(f"Invalid records: {invalid_df.count()}")

if invalid_df.count() > 0:
    display(invalid_df)
    raise ValueError("Validation failed. Data was not loaded.")

Load into the Bronze Delta table

In [0]:
# if not DRY_RUN:
#     (
#         df.write
#         .format("delta")
#         .mode("overwrite")
#         .option("overwriteSchema", "true")
#         .saveAsTable(TARGET_TABLE)
#     )

#     print(f"Successfully loaded data into {TARGET_TABLE}")
# else:
#     print("DRY RUN: Data validated, but no table was written.")

In [0]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

# Audit configuration
AUDIT_TABLE = "tx_dev.control.tax_ingestion_audit"

run_id = str(uuid.uuid4())
start_time = datetime.now(timezone.utc)

source_record_count = df.count()
target_record_count = 0
status = "STARTED"
error_message = None

try:
    if not DRY_RUN:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(TARGET_TABLE)
        )

        target_record_count = spark.table(TARGET_TABLE).count()

    status = "SUCCESS"

except Exception as e:
    status = "FAILED"
    error_message = str(e)[:4000]

finally:
    end_time = datetime.now(timezone.utc)

    audit_data = [(
        run_id,
        SOURCE_SYSTEM,
        SOURCE_PATH.split("/")[-1],
        TARGET_TABLE,
        LOAD_TYPE,
        start_time,
        end_time,
        source_record_count,
        target_record_count,
        status,
        error_message
    )]

    audit_schema = """
        run_id STRING,
        source_system STRING,
        source_file_name STRING,
        target_table STRING,
        load_type STRING,
        start_time TIMESTAMP,
        end_time TIMESTAMP,
        source_record_count BIGINT,
        target_record_count BIGINT,
        status STRING,
        error_message STRING
    """

    audit_df = spark.createDataFrame(
        audit_data,
        schema=audit_schema
    )

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(AUDIT_TABLE)
    )

if status == "FAILED":
    raise RuntimeError(
        f"Ingestion failed. Run ID: {run_id}. "
        f"Error: {error_message}"
    )

print(f"Run ID: {run_id}")
print(f"Status: {status}")
print(f"Source records: {source_record_count}")
print(f"Target records: {target_record_count}")

VALIDATION AND VERIFY 

In [0]:
# %sql
# SELECT *
# FROM tx_dev.bronze.brz_tax_obligations;

In [0]:
%sql
---CHECK AUDIT RECORD
SELECT
    source_file_name,
    target_table,
    source_record_count,
    target_record_count,
    status,
    start_time
FROM tx_dev.control.tax_ingestion_audit
ORDER BY start_time DESC;